# Notebook 03: Training the Construction Classifier

Walks through:
1. Dataset preparation and augmentation
2. Fine-tuning MobileNetV3-small
3. Evaluating on the validation split
4. Comparing CNN vs rule-based results

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_DIR = Path('../ml/data/construction')
print(f'Data directory exists: {DATA_DIR.exists()}')
print(f'Train positive: {len(list((DATA_DIR / "train/positive").glob("*") if (DATA_DIR / "train/positive").exists() else []))} images')
print(f'Train negative: {len(list((DATA_DIR / "train/negative").glob("*") if (DATA_DIR / "train/negative").exists() else []))} images')

In [ ]:
# Visualise sample patches from the dataset
if (DATA_DIR / 'train').exists():
    transform_viz = transforms.Compose([transforms.Resize((128, 128)), transforms.ToTensor()])
    ds = datasets.ImageFolder(DATA_DIR / 'train', transform=transform_viz)
    
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for cls_idx, cls_name in enumerate(ds.classes):
        cls_indices = [i for i, (_, label) in enumerate(ds.samples) if label == cls_idx]
        for j in range(min(8, len(cls_indices))):
            img, _ = ds[cls_indices[j]]
            axes[cls_idx, j].imshow(img.permute(1, 2, 0).numpy())
            axes[cls_idx, j].set_title(cls_name if j == 0 else '', fontsize=8)
            axes[cls_idx, j].axis('off')
    plt.tight_layout()
    plt.savefig('03_sample_patches.png', dpi=150)
    plt.show()
else:
    print('Dataset not yet available. Run ml/train_construction.py after preparing data.')

In [ ]:
# Launch training (runs synchronously in the notebook)
# Alternatively, run: python ml/train_construction.py --epochs 20

if (DATA_DIR / 'train').exists():
    from ml.train_construction import train
    model_path = train(epochs=5, batch_size=16)  # short run for notebook demo
    print(f'Model saved: {model_path}')
else:
    print('Skipping training — data not available')

In [ ]:
# Evaluate the best model
from pathlib import Path
import subprocess, sys

models_dir = Path('../models')
construction_models = sorted(models_dir.glob('construction_v*.pt'))

if construction_models:
    best_model = construction_models[-1]
    print(f'Evaluating: {best_model}')
    result = subprocess.run(
        [sys.executable, '../ml/eval.py', '--model', str(best_model), '--class', 'construction'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[:500])
else:
    print('No trained models found at ../models/')